# LangGraph 03 — Hugging Face LLM chạy local trên Colab

`START → summarize → create_quiz → quality_check → (revise_quiz | END)`

## 1. Khởi tạo model local

`Qwen/Qwen2.5-0.5B-Instruct` được chọn vì có chat template, làm theo instruction/format tốt hơn FLAN-T5 nhỏ, hỗ trợ đa ngôn ngữ và vẫn đủ nhẹ để chạy trên CPU Colab. Nếu có GPU, có thể đổi sang bản 1.5B để tăng chất lượng.

In [ ]:
import os
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_huggingface import ChatHuggingFace, HuggingFacePipeline

HF_MODEL = os.getenv("HF_MODEL", "Qwen/Qwen2.5-0.5B-Instruct")
# base_llm
base_llm = HuggingFacePipeline.from_model_id(
    model_id=HF_MODEL,
    task="text-generation",
    pipeline_kwargs={
        "max_new_tokens": 180,
        "do_sample": False,
        "repetition_penalty": 1.1,
        "return_full_text": False
    },
    
)

# local_llm
local_llm = ChatHuggingFace(
    llm=base_llm,
)

print("Đã tải:", HF_MODEL)

/Users/vunguyen/Documents/Mine/Python_Cyber/Lesson16/myenv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Device set to use mps:0
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Đã tải: Qwen/Qwen2.5-0.5B-Instruct


## 2. State và các node

`attempts` là guard chống loop vô hạn. Quality check không dùng LLM: nó kiểm tra format đầu ra một cách có thể dự đoán và test được.

In [ ]:
import re
from typing import Literal, TypedDict
from langgraph.graph import END, START, StateGraph

class ContentState(TypedDict, total=False):
    source: str
    summary: str
    quiz: str
    quiz_ok: bool
    feedback: str
    attempts: int

def summarize(state: ContentState):
    response = local_llm.invoke([
        SystemMessage(content="You are a precise teaching assistant. Use only facts in the source."),
        HumanMessage(content=(
            "Summarize the source in 3 to 5 concise bullet points. "
            "Keep important technical terms.\n\nSOURCE:\n" + state["source"]
        )),
    ])
    return {"summary": response.content.strip()}

def create_quiz(state: ContentState):
    feedback = state.get("feedback", "")
    response = local_llm.invoke([
        SystemMessage(content="You create quizzes using only the supplied summary."),
        HumanMessage(content=(
            "Create exactly 3 numbered short-answer questions. Put one question per line. "
            "Use only this format: '1. ...?', '2. ...?', '3. ...?'. "
            f"Revision feedback: {feedback}\n\nSUMMARY:\n{state['summary']}"
        )),
    ])
    return {
        "quiz": response.content.strip(),
        "attempts": state.get("attempts", 0) + 1,
    }

def quality_check(state: ContentState):
    numbers = re.findall(r"(?m)^\s*([1-3])[.):-]", state["quiz"])
    ok = set(numbers) == {"1", "2", "3"}
    feedback = "" if ok else "Output must contain exactly three numbered questions, one per line."
    return {"quiz_ok": ok, "feedback": feedback}

def after_check(state: ContentState) -> Literal["revise", "done"]:
    if state["quiz_ok"] or state["attempts"] >= 2:
        return "done"
    return "revise"

In [ ]:
builder = StateGraph(ContentState)
builder.add_node("summarize", summarize)
builder.add_node("create_quiz", create_quiz)
builder.add_node("quality_check", quality_check)
builder.add_edge(START, "summarize")
builder.add_edge("summarize", "create_quiz")
builder.add_edge("create_quiz", "quality_check")
builder.add_conditional_edges(
    "quality_check",
    after_check,
    {"revise": "create_quiz", "done": END},
)

content_graph = builder.compile()
print(content_graph.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	summarize(summarize)
	create_quiz(create_quiz)
	quality_check(quality_check)
	__end__([<p>__end__</p>]):::last
	__start__ --> summarize;
	create_quiz --> quality_check;
	quality_check -. &nbsp;done&nbsp; .-> __end__;
	quality_check -. &nbsp;revise&nbsp; .-> create_quiz;
	summarize --> create_quiz;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



In [ ]:
source = (
    "LangGraph is a low-level orchestration framework for building stateful agents. "
    "A graph contains nodes that perform work and edges that determine control flow. "
    "State is shared between nodes. Conditional edges enable routing and loops. "
    "Checkpointers persist state so applications can support memory and human review."
)

final = None
for state in content_graph.stream({"source": source}, stream_mode="values"):
    final = state
    print("State hiện có:", list(state))

print("\nSUMMARY\n", final["summary"])
print("\nQUIZ\n", final["quiz"])
print("\nPassed format check:", final["quiz_ok"], "| attempts:", final["attempts"])

State hiện có: ['source']
State hiện có: ['source', 'summary']
State hiện có: ['source', 'summary', 'quiz', 'attempts']
State hiện có: ['source', 'summary', 'quiz', 'quiz_ok', 'feedback', 'attempts']

SUMMARY
 - **Language**: Low-level orchestration framework
- **Functionality**: Graph-based, stateful agent architecture
- **Nodes**: Perform work and share state
- **Edges**: Determine control flow and enable conditional routing
- **Checkpointing**: Persist state across checkpoints for memory management and human review
- **Architecture**: Memory-efficient, supports human review through checkpointing

QUIZ
 1. What is the primary purpose of a graph-based, stateful agent architecture in this framework?
2. How does the functionality of performing work and sharing state differ from other frameworks mentioned?
3. Describe the mechanism by which the checkpointing feature ensures that state persists across checkpoints.

Passed format check: True | attempts: 1
